# Extracción de características
En este dataset nos vamos a centrar en la extracción de las características de la voz. Esto nos servirá para que un modelo de IA clasifique si un audio es generado o es natural y, posteriormente, para que un modelo de XAI explique la decisión de dicho modelo.

## 1 Imports
El primer paso es declarar qué librerías vamos a usar para esta tarea.

In [1]:
import os
import librosa
import numpy as np
import pandas as pd
import parselmouth
from scipy.stats import skew, kurtosis
from tqdm import tqdm
import json

## 2 Rutas
Ahora definimos las rutas donde tenemos los datos. Dichas rutas pueden cambiar y el código seguiría funcionando correctamente.

In [ ]:
SPOOF = 'Dataset\\Generado\\Español'
OUTPUT_SPOOF = 'Features\\spoof_features.csv'
BONAFIDE = 'Dataset\\Natural\\Español'
OUTPUT_BONAFIDE = 'Features\\bonafide_features.csv'

## 3 Extraer archivos
Antes de sacar las características necesitamos la ruta de los archivos, lo cual podemos obtener fácilmente recorriendo los directorios.

In [3]:
def get_files(dir):
    all_audio_files = []
    for root, _, files in os.walk(dir):
        for file in files:
            if file.lower().endswith(".wav"):
                all_audio_files.append(os.path.join(root, file))
    return all_audio_files

In [4]:
bonafide_files = get_files(BONAFIDE)
spoof_files = get_files(SPOOF)

In [5]:
bonafide_files

['Dataset\\Natural\\Español\\E_10_N.wav',
 'Dataset\\Natural\\Español\\E_11_N.wav',
 'Dataset\\Natural\\Español\\E_12_N.wav',
 'Dataset\\Natural\\Español\\E_13_N.wav',
 'Dataset\\Natural\\Español\\E_14_N.wav',
 'Dataset\\Natural\\Español\\E_15_N.wav',
 'Dataset\\Natural\\Español\\E_16_N.wav',
 'Dataset\\Natural\\Español\\E_17_N.wav',
 'Dataset\\Natural\\Español\\E_18_N.wav',
 'Dataset\\Natural\\Español\\E_19_N.wav',
 'Dataset\\Natural\\Español\\E_1_N.wav',
 'Dataset\\Natural\\Español\\E_20_N.wav',
 'Dataset\\Natural\\Español\\E_21_N.wav',
 'Dataset\\Natural\\Español\\E_22_N.wav',
 'Dataset\\Natural\\Español\\E_23_N.wav',
 'Dataset\\Natural\\Español\\E_24_N.wav',
 'Dataset\\Natural\\Español\\E_25_N.wav',
 'Dataset\\Natural\\Español\\E_26_N.wav',
 'Dataset\\Natural\\Español\\E_27_N.wav',
 'Dataset\\Natural\\Español\\E_28_N.wav',
 'Dataset\\Natural\\Español\\E_29_N.wav',
 'Dataset\\Natural\\Español\\E_2_N.wav',
 'Dataset\\Natural\\Español\\E_30_N.wav',
 'Dataset\\Natural\\Español\\E_31_N.

## 4 Extraer características
Ahora viene el punto clave: qué características queremos y cómo las sacamos. En total tenemos 28 características. Vamos a ver cada una de ellas: qué mide, qué representa y cómo las extraemos de un audio.
- Root Mean Square (rms): 
    - Valor cuadrático medio. 
    - Mide: La energía promedio de la señal de audio. 
    - Representa: La intensidad o volumen general del audio.
    - Extracción: 
        1. Se divide la señal en frames (marcos temporales). 
        2. Para cada frame, se calcula la raíz cuadrada del promedio de los valores al cuadrado.
        3. Se promedian todos los valores de RMS de los frames.
    - Información extra: En voz humana natural, el RMS tiende a tener variaciones más orgánicas, mientras que audios sintéticos pueden mostrar patrones más uniformes.
- Zero Cross Rate (zcr):
    - Tasa de cruces por cero.
    - Mide: El número de veces que la señal cambia de signo por segundo.
    - Representa: El contenido de frecuencias altas y la sonoridad.
    - Extracción:
        1. Se detectan los puntos donde la señal cambia de positivo a negativo o viceversa.
        2. Se cuenta el número de cruces por unidad de tiempo.
        3. Se normaliza por la longitud de la ventana.
    - Información extra: Los sonidos sordos (como /s/, /f/) tienen ZCR alto, mientras los sonoros (/a/, /m/) tienen ZCR bajo.
- spectral centroid:
    - Centroide espectral.
    - Mide: El centro de masa del espectro de frecuencia. Indica dónde se concentra la energía espectral.
    - Representa: El "brillo" o agudeza del sonido.
    - Extracción:
        1. Se calcula la FFT para obtener el espectro de frecuencia.
        2. Se ponderan las frecuencias por su magnitud.
        3. Se calcula el promedio ponderado de las frecuencias.
    - Información extra: Voces femeninas tienden a tener centroides más altos que masculinas.
- spectral bandwith:
    - Ancho de banda espectral.
    - Mide: La dispersión del espectro alrededor del centroide.
    - Representa: La variedad de frecuencias presentes en la voz.
    - Extracción:
        1. Se calcula el centroide espectral.
        2. Se mide la desviación estándar del espectro respecto al centroide.
        3. Se expresa en Hz.
- spectral rolloff:
    - Rolloff espectral.
    - Mide: La frecuencia por debajo de la cual se encuentra un porcentaje (85% por defecto) de la energía espectral.
    - Representa: El límite superior del contenido frecuencial dominante.
    - Extracción:
        1. Obtener la FFT de cada frame para ver la distribución de energía en frecuencias.
        2. Sumar progresivamente la energía desde 0 Hz hasta cubrir el porcentaje umbral (85%).
        3. Identificar la frecuencia donde se alcanza ese umbral de energía acumulada.
    - Información extra: Útil para distinguir entre sonidos con energía concentrada en bajas frecuencias vs. amplio espectro.
- spectral flatness:
    - Planitud espectral.
    - Mide: Cuán similar es el espectro a ruido blanco (plano).
    - Representa: la distinción entre sonidos tonales (baja flatness) y ruidosos (alta flatness).
    - Extracción:
        1. Se calcula la media geométrica del espectro.
        2. Se calcula la media aritmética del espectro.
        3. Se divide la media geométrica entre la aritmética.
- pitch mean: 
    - Tono fundamental (F0) - Media.
    - Mide: La frecuencia promedio de vibración de las cuerdas vocales.
    - Representa: El tono promedio de la voz.
    - Extracción:
        1. pYIN obtiene las frecuencias fundamentales aplicando el algoritmo YIN.
        2. Después se aplica el algoritmo de Viterbi para obtener la cadena de F0 más probable.
        3. Se calcula la media usando np.nanmean.
- pitch standard deviation:
    - Tono fundamental (F0) - Desviación típica.
    - Mide:
- formants 1-3:
    - Formantes (F1, F2, F3).
    - Mide: Las frecuencias de resonancia del tracto vocal que definen los sonidos vocálicos.
    - Representa: F1 relacionado con apertura bucal, F2 con posición de lengua, F3 con calidad vocal.
    - Extracción:
        1. Se usa el método de Burg para estimar formantes.
        2. Se muestrean 100 puntos a lo largo del audio.
        3. Se calcula la media usando np.nanmean.
- Harmonic-to-Noise Ratio (hnr):
    - Relación Armónico-Ruido.
    - Mide: La proporción entre energía periódica (armónicos) y energía aperiódica (ruido).
    - Representa: La calidad vocal y presencia de ruido glotal.
    - Extracción:
        1. Dividir la señal en parte periódica (armónicos) y aperiódica (ruido).
        2. Medir la energía de cada componente por separado.
        3. Dividir energía armónica entre energía de ruido y convertir a decibelios.
    - Información extra: Valores bajos pueden indicar voz ronca o patológica. Voces humanas naturales suelen tener HNR entre 10-20 dB; valores extremos pueden indicar síntesis.
- silence ratio:
    - Proporción de silencio.
    - Mide: El porcentaje de frames considerados silencio.
    - Representa: Los patrones de pausas y continuidad del habla.
    - Extracción:
        1. Calcular RMS para segmentos cortos del audio.
        2. Comparar cada valor con un umbral mínimo (ej: 0.02).
        3. Calcular porcentaje de frames que caen bajo el umbral.
    - Información extra: Las IAs pueden generar patrones de pausas poco naturales o demasiado regulares.
- Mel Frequency cepstral coefficients (mfcc) 1-13:
    - Coeficientes Cepstrales en Frecuencia Mel.
    - Mide: Representación compacta de la envolvente espectral en escala perceptual Mel.
    - Representa: MFCC1 relacionado con energía, otros con forma del tracto vocal.
    - Extracción:
        1. Aplicar filtros triangulares que simulan percepción humana de frecuencia.
        2. Comprimir el rango dinámico calculando logaritmo de las energías.
        3. Transformada Coseno Discreta para obtener coeficientes cepstrales.
    - Información extra: Estándar en reconocimiento de voz; muy efectivos para capturar timbre.
- skewness:
    - Asimetría.
    - Mide: La medida de la asimetría de la distribución de amplitud.
    - Representa: El sesgo en la forma de onda de la voz.
    - Extracción:
        1. Restar la media a cada muestra y elevar al cubo.
        2. Calcular el promedio de estas desviaciones cúbicas.
        3. Dividir por la desviación estándar al cubo para estandarizar.
    - Información extra: Distribuciones de amplitud asimétricas pueden indicar características anómalas en voces sintéticas.
- kurtosis:
    - Curtosis.
    - Mide: La medida de la "pesadez" de las colas de la distribución.
    - Representa: La presencia de picos pronunciados en la señal.
    - Extracción:
        1. Elevar las desviaciones respecto a la media a la cuarta potencia.
        2. Calcular el promedio de estos valores.
        3. Restar 3 para que la distribución normal tenga curtosis = 0.
    - Información extra: Curtosis alta (distribución picuda) puede aparecer en audios con compresión excesiva o artefactos de síntesis.

In [6]:
def extract_pitch(audio, sr):
    f0, voiced_flag, voiced_prob = librosa.pyin(
            y=audio, 
            fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'),
            sr=sr
        )

    f0_sonoro = f0[voiced_flag]
    
    if len(f0_sonoro) == 0:
        return None  # No hay pitch detectable

    return f0_sonoro

def extract_formants(path):
    snd = parselmouth.Sound(path)
    formant = snd.to_formant_burg()
    f1, f2, f3 = [], [], []
    for t in np.linspace(0, snd.duration, 100):
        try:
            f1.append(formant.get_value_at_time(1, t))
            f2.append(formant.get_value_at_time(2, t))
            f3.append(formant.get_value_at_time(3, t))
        except:
            pass
    return f1, f2, f3

def extract_silence(audio, sr, threshold=0.02):
    energy = librosa.feature.rms(y=audio)[0]
    silence_ratio = np.mean(energy < threshold)
    return silence_ratio

def extract_hnr(path):
    snd = parselmouth.Sound(path)
    hnr = snd.to_harmonicity()
    return np.mean(hnr.values[hnr.values != -200])

In [7]:
def extract_features(path):
    try:
        audio, sr = librosa.load(path, sr=None)
        duration = librosa.get_duration(y=audio, sr=sr)
        
        rms = np.mean(librosa.feature.rms(y=audio))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio))
        centroid = np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))
        bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr))
        rolloff = np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr))
        flatness = np.mean(librosa.feature.spectral_flatness(y=audio))
        
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

        pitch = extract_pitch(audio, sr)
        pitch_mean = np.nanmean(pitch)
        pitch_std = np.nanstd(pitch)
        
        f1, f2, f3 = extract_formants(path)
        f1_mean = np.nanmean(f1)
        f2_mean = np.nanmean(f2)
        f3_mean = np.nanmean(f3)
        
        hnr = extract_hnr(path)
        silence = extract_silence(audio, sr)
        
        features = {
            "filename": path,
            "duration": duration,
            "rms": rms,
            "zcr": zcr,
            "spectral_centroid": centroid,
            "spectral_bandwidth": bandwidth,
            "spectral_rolloff": rolloff,
            "spectral_flatness": flatness,
            "pitch": pitch,
            "pitch_mean": pitch_mean,
            "pitch_std": pitch_std,
            "f1": f1,
            "f2": f2,
            "f3": f3,
            "f1_mean": f1_mean,
            "f2_mean": f2_mean,
            "f3_mean": f3_mean,
            "hnr": hnr,
            "silence_ratio": silence,
        }

        for i, val in enumerate(mfcc):
            features[f"mfcc_{i+1}"] = val
            features[f"mfcc_{i+1}_mean"] = np.nanmean(val)
        
        # Estadísticos globales
        features["skewness"] = skew(audio)
        features["kurtosis"] = kurtosis(audio)
        
        return features
    except Exception as e:
        print(f"Error procesando {path}: {e}")
        return None

In [8]:
def features(files):
    features_list = []
    for file in tqdm(files, desc="Extrayendo características"):
        feats = extract_features(file)
        if feats:
            features_list.append(feats)
    return features_list

In [9]:
df_spoof = pd.DataFrame(features(spoof_files))
df_bonafide = pd.DataFrame(features(bonafide_files))

Extrayendo características:   0%|          | 0/50 [00:00<?, ?it/s]

Extrayendo características: 100%|██████████| 50/50 [00:36<00:00,  1.36it/s]


Ahora tenemos que poner las etiquetas de los audios y guardar las listas con json.dumps para que al cargar el csv dichas listas mantengan el formato, porque por defecto pandas.to_csv parece que convierte las listas a strings.

In [10]:
df_spoof['label'] = 0
df_spoof['pitch'] = df_spoof['pitch'].apply(lambda x: json.dumps(x.tolist()))
df_spoof['f1'] = df_spoof['f1'].apply(lambda x: json.dumps(x))
df_spoof['f2'] = df_spoof['f2'].apply(lambda x: json.dumps(x))
df_spoof['f3'] = df_spoof['f3'].apply(lambda x: json.dumps(x))
for col in df_spoof.columns:
    if 'mfcc' in col and not col.endswith('mean'):
        df_spoof[col] = df_spoof[col].apply(lambda x: json.dumps(x.tolist()))

In [11]:
df_bonafide['label'] = 1
df_bonafide['pitch'] = df_bonafide['pitch'].apply(lambda x: json.dumps(x.tolist()))
df_bonafide['f1'] = df_bonafide['f1'].apply(lambda x: json.dumps(x))
df_bonafide['f2'] = df_bonafide['f2'].apply(lambda x: json.dumps(x))
df_bonafide['f3'] = df_bonafide['f3'].apply(lambda x: json.dumps(x))
for col in df_bonafide.columns:
    if 'mfcc' in col and not col.endswith('mean'):
        df_bonafide[col] = df_bonafide[col].apply(lambda x: json.dumps(x.tolist()))

## 5 Guardar los datos
Para finalizar guardamos los datos en un csv, lo cual nos permitirá usarlos más adelante de manera fácil y rápida.

In [12]:
df_spoof.to_csv(OUTPUT_SPOOF, index=False)
df_bonafide.to_csv(OUTPUT_BONAFIDE, index=False)

## 6 Ejemplo de cargar los datos
Para terminar hacemos un ejemplo de como cargar los datos, ya que al guardar los arrays como el tono fundamental tuvimos que aplicarle una transformación para que se guarde de tal manera que al cargarse tengamos el array como antes de guardarlo. Si se usan las características, aplicar el código que aparece abajo.

In [ ]:
spoof = pd.read_csv(OUTPUT_SPOOF)
bonafide = pd.read_csv(OUTPUT_BONAFIDE)

In [ ]:
df = pd.concat([spoof, bonafide])

In [ ]:
df['pitch'] = df['pitch'].apply(lambda x: np.array(json.loads(x)))
df['f1'] = df['f1'].apply(lambda x: json.loads(x))
df['f2'] = df['f2'].apply(lambda x: json.loads(x))
df['f3'] = df['f3'].apply(lambda x: json.loads(x))
for col in df.columns:
    if 'mfcc' in col and not col.endswith('mean'):
        df[col] = df[col].apply(lambda x: np.array(json.loads(x)))